# Flag non-vegetation endmembers (for manual removal)

Input can be either:
* the CSV written by `smacc_endmember_extraction.ipynb` (columns `class`, `endmember_no`, ..., one column per wavelength), or
* an ENVI spectral library (`.sli` + `.hdr`), e.g. endmembers exported from ENVI's SMACC tool. Each library is treated as one class.

Every endmember is tested against simple spectral rules. The result is a table with a `flag` (`REMOVE` / `CHECK` / `ok`), the reasons, and a `keep` column.
**You make the final decision.** Open `endmembers_flagged.csv` in Excel, look at the plots, set `keep` to `False` for the spectra you reject, then run the last cell to write the cleaned file.

| Rule | Typical cause | Level |
|---|---|---|
| NDVI < `NDVI_MIN` | soil, water, dry residue, strongly mixed pixel | REMOVE |
| R800 / R670 < `SR_MIN` | no red-edge, so not green canopy | REMOVE |
| NDWI = (R560 − R860)/(R560 + R860) > 0 | standing water | REMOVE |
| R800 < `NIR_MIN` | deep shadow, or water | REMOVE |
| Mean visible (400–700 nm) > `VIS_MAX` | specular glint, bright soil, or a white object | CHECK |
| No green peak (R550 ≤ R670) | brown lesion or senescent leaf, **or** soil | CHECK |
| Spike: a band deviates from its 5-band running median by > `SPIKE_MIN` | sensor noise, bad pixel | CHECK |

⚠️ **Do not throw away every CHECK spectrum from the diseased class.** Brown, necrotic or yellow leaves *are* the disease signal. They often lose the green peak and have a lower NDVI while still showing a red-edge. Remove a CHECK spectrum only if its shape and location (the plots below) show it is not plant material.

In [ ]:
# ===================== USER SETTINGS =====================
INPUT   = r"D:\Coalighned\24_02_2026\ROI(TRIAL)\TRIAL_h_d_combined\smacc_endmembers_healthy_diseased.csv"    # CSV from the SMACC notebook, or a .sli / .hdr spectral library
CLASS_LIBS = None     # for ENVI libraries only: e.g. {"healthy": r"...healthy.sli", "diseased": r"...diseased.sli"} (then INPUT is ignored)
OUT_DIR = r"D:\Coalighned\24_02_2026\ROI(TRIAL)\TRIAL_h_d_combined"

NDVI_MIN = 0.30       # below this: REMOVE
SR_MIN   = 2.0        # R800/R670 below this: REMOVE (no red-edge)
NIR_MIN  = 0.10       # R800 below this (reflectance 0-1): REMOVE (shadow / water)
VIS_MAX  = 0.25       # mean 400-700 nm above this (reflectance 0-1): CHECK (glint / bright soil)
SPIKE_MIN = 0.03      # a band deviating this much (reflectance 0-1) from its 5-band running median: CHECK

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(OUT_DIR, exist_ok=True)

def load_sli(path, cls):
    import spectral.io.envi as envi
    hdr = os.path.splitext(path)[0] + ".hdr"
    sli = path if not path.lower().endswith(".hdr") else None
    lib = envi.open(hdr, sli)
    wl = np.array([float(w) for w in lib.bands.centers])
    names = lib.names if lib.names else [f"{cls}_{i+1}" for i in range(len(lib.spectra))]
    meta = pd.DataFrame({"class": cls, "endmember_no": np.arange(1, len(names) + 1), "name": names})
    return pd.concat([meta, pd.DataFrame(lib.spectra, columns=[f"{w:.2f}" for w in wl])], axis=1)

if CLASS_LIBS:
    df = pd.concat([load_sli(p, c) for c, p in CLASS_LIBS.items()], ignore_index=True)
elif INPUT.lower().endswith((".sli", ".hdr")):
    df = load_sli(INPUT, os.path.splitext(os.path.basename(INPUT))[0])
else:
    df = pd.read_csv(INPUT)

def is_wavelength(c):
    return re.fullmatch(r"\d+(\.\d+)?", str(c).strip()) is not None
band_cols = [c for c in df.columns if is_wavelength(c)]
meta_cols = [c for c in df.columns if c not in band_cols]
wl = np.array([float(c) for c in band_cols])
X = df[band_cols].to_numpy(dtype=float)

# Reflectance scale: brightness thresholds are for 0-1 reflectance; detect 0-100 or 0-10000 data
p99 = np.nanpercentile(X, 99)
scale = 10000.0 if p99 > 100 else (100.0 if p99 > 2 else 1.0)
R = X / scale
print(f"{len(df)} spectra, {len(wl)} bands ({wl.min():.0f}-{wl.max():.0f} nm), reflectance scale 1/{scale:g}")
print(df["class"].value_counts().to_string())

## Apply the rules

In [ ]:
def b(nm):
    return R[:, int(np.argmin(np.abs(wl - nm)))]

def band_mean(lo, hi):
    m = (wl >= lo) & (wl <= hi)
    return R[:, m].mean(axis=1)

with np.errstate(divide="ignore", invalid="ignore"):
    red, nir, green, nir860 = band_mean(660, 680), band_mean(790, 810), band_mean(545, 555), band_mean(850, 870)
    ndvi = (nir - red) / (nir + red)
    sr = nir / red
    ndwi = (green - nir860) / (green + nir860)
vis = band_mean(400, 700)

# Spikes: largest deviation from a 5-band running median, within each contiguous block of bands
from numpy.lib.stride_tricks import sliding_window_view
d = np.diff(wl)
neighbour = np.maximum(np.r_[d[1], d[:-1]], np.r_[d[1:], d[-2]])
block = np.r_[0, np.cumsum(d > 2.5 * neighbour)]          # bands separated by removed water bands are different blocks
spike = np.zeros(len(R))
for k in np.unique(block):
    Rb = R[:, block == k]
    if Rb.shape[1] < 5:
        continue
    med = np.median(sliding_window_view(np.pad(Rb, ((0, 0), (2, 2)), mode="edge"), 5, axis=1), axis=2)
    spike = np.maximum(spike, np.abs(Rb - med).max(axis=1))

rules = [  # (mask, level, reason)
    (ndvi < NDVI_MIN,      "REMOVE", f"NDVI<{NDVI_MIN} (soil/water/residue/mixed)"),
    (sr < SR_MIN,          "REMOVE", f"R800/R670<{SR_MIN} (no red-edge)"),
    (ndwi > 0,             "REMOVE", "NDWI>0 (water)"),
    (nir < NIR_MIN,        "REMOVE", f"R800<{NIR_MIN} (shadow/water)"),
    (vis > VIS_MAX,        "CHECK",  f"mean VIS>{VIS_MAX} (glint/bright soil?)"),
    (green <= red,         "CHECK",  "no green peak (lesion/senescent leaf or soil)"),
    (spike > SPIKE_MIN,    "CHECK",  "spiky spectrum (noise)"),
]
reasons = [[] for _ in range(len(R))]
level = np.array(["ok"] * len(R), dtype=object)
for mask, lev, why in rules:
    mask = np.nan_to_num(mask, nan=True).astype(bool)
    for i in np.nonzero(mask)[0]:
        reasons[i].append(why)
        if lev == "REMOVE" or level[i] == "ok":
            level[i] = lev
bad_values = ~np.isfinite(R).all(axis=1)
for i in np.nonzero(bad_values)[0]:
    reasons[i].append("NaN/inf values"); level[i] = "REMOVE"

flags = pd.DataFrame({"flag": level, "reasons": ["; ".join(r) for r in reasons],
                      "NDVI": ndvi.round(3), "SR_800_670": sr.round(2), "NDWI": ndwi.round(3),
                      "R800": nir.round(4), "VIS_mean": vis.round(4), "max_spike": spike.round(4)})
flags["keep"] = flags["flag"] != "REMOVE"          # CHECK spectra are kept until you decide
out = pd.concat([df[meta_cols], flags, df[band_cols]], axis=1)

print(pd.crosstab(out["class"], out["flag"]))
flag_path = os.path.join(OUT_DIR, "endmembers_flagged.csv")
out.to_csv(flag_path, index=False)
print("\nSaved:", flag_path)
id_cols = [c for c in ["class", "endmember_no", "name", "row_0based", "col_0based"] if c in out.columns]
out.loc[out["flag"] != "ok", id_cols + ["flag", "reasons", "NDVI", "SR_800_670", "R800"]]

## Look at them

In [ ]:
def gapped(y):
    gap = np.where(d > 2.5 * neighbour)[0] + 1
    return np.insert(wl, gap, np.nan), np.insert(np.asarray(y, float), gap, np.nan)

classes = list(out["class"].unique())
colours = {"ok": "tab:green", "CHECK": "tab:orange", "REMOVE": "tab:red"}
fig, axes = plt.subplots(1, len(classes), figsize=(8 * len(classes), 5), squeeze=False)
for ax, cls in zip(axes[0], classes):
    for lev in ["ok", "CHECK", "REMOVE"]:
        rows = np.nonzero(((out["class"] == cls) & (out["flag"] == lev)).to_numpy())[0]
        for j, i in enumerate(rows):
            ax.plot(*gapped(R[i]), color=colours[lev], lw=0.8 if lev != "ok" else 0.5,
                    alpha=0.9 if lev != "ok" else 0.15, label=f"{lev} ({len(rows)})" if j == 0 else None)
    ax.set_title(f"{cls}"); ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Reflectance (0-1)"); ax.legend()
plt.tight_layout(); plt.show()

# One small panel per flagged spectrum, so each can be judged individually
MAX_PANELS = 120
flagged = np.r_[np.nonzero((out["flag"] == "REMOVE").to_numpy())[0], np.nonzero((out["flag"] == "CHECK").to_numpy())[0]]
if len(flagged) > MAX_PANELS:
    print(f"{len(flagged)} flagged spectra; showing the first {MAX_PANELS} (REMOVE first). Raise MAX_PANELS to see all.")
    flagged = flagged[:MAX_PANELS]
if len(flagged):
    ncol = 6
    nrow = int(np.ceil(len(flagged) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.4 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), flagged):
        ax.axis("on")
        ax.plot(*gapped(R[i]), color=colours[out.at[i, "flag"]])
        tag = f"{out.at[i, 'class']} #{out.at[i, 'endmember_no']}" if "endmember_no" in out else str(i)
        ax.set_title(f"{tag}  NDVI={out.at[i, 'NDVI']:.2f}", fontsize=8)
        ax.tick_params(labelsize=6)
    plt.tight_layout(); plt.show()
else:
    print("No spectra flagged.")

## Your decision, then write the cleaned file

Either edit the `keep` column of `endmembers_flagged.csv` in Excel and save it, **or** list the endmembers to drop or rescue here. Then run the cell.

In [ ]:
MANUAL_REMOVE = {}   # e.g. {"healthy": [12, 57], "diseased": [3]}     endmember_no values to drop
MANUAL_KEEP   = {}   # e.g. {"diseased": [41, 88]}                      REMOVE-flagged ones you want to keep

final = pd.read_csv(flag_path)            # re-read, so edits made in Excel are used
key = "endmember_no" if "endmember_no" in final.columns else None
for cls, nums in MANUAL_REMOVE.items():
    final.loc[(final["class"] == cls) & final[key].isin(nums), "keep"] = False
for cls, nums in MANUAL_KEEP.items():
    final.loc[(final["class"] == cls) & final[key].isin(nums), "keep"] = True

clean = final[final["keep"].astype(str).str.lower().isin(["true", "1"])]
clean_path = os.path.join(OUT_DIR, "endmembers_vegetation_only.csv")
clean.to_csv(clean_path, index=False)
print(pd.crosstab(final["class"], final["keep"]).rename(columns={True: "kept", False: "removed"}))
print("Saved:", clean_path)